# 08 — Evaluation: Base vs Fine-tuned Comparison

Loads all fine-tuned adapters and runs the same eval as notebook 00.
Produces the final comparison table for your defense.

In [ ]:
import sys
sys.path.append('/kaggle/working/final_year_proj_finetuning')

from unsloth import FastLanguageModel
from src.data_utils import load_jsonl
from src.evaluation import evaluate_translation, evaluate_summarization, evaluate_qa, build_results_table
from src.utils import set_seed, print_gpu_memory
import pandas as pd
import matplotlib.pyplot as plt
import torch, gc

set_seed(42)

val_tr  = load_jsonl('data/processed/translation_val.jsonl')
val_sum = load_jsonl('data/processed/summarization_val.jsonl')
val_qa  = load_jsonl('data/processed/qa_val.jsonl')

# Load baseline results
baseline_df = pd.read_csv('results/base_vs_finetuned/baseline_results.csv')
all_results = baseline_df.set_index('Model').to_dict(orient='index')
print('Loaded baseline results:')
print(baseline_df)

In [ ]:
def eval_adapter(adapter_repo: str, model_key_size: str, label: str):
    """Load fine-tuned adapter from HF and evaluate."""
    print(f'\n{"-"*60}')
    print(f'Evaluating: {label}')
    print(f'Adapter: {adapter_repo}')
    print(f'{"-"*60}')

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_repo,
        max_seq_length=512,
        load_in_4bit=True,
    )

    tr_res  = evaluate_translation(model, tokenizer, val_tr, n_samples=100)
    sum_res = evaluate_summarization(model, tokenizer, val_sum, n_samples=100)
    qa_res  = evaluate_qa(model, tokenizer, val_qa, n_samples=100)

    result = {
        'BLEU': tr_res['bleu'],
        'ROUGE-1': sum_res['rouge1'],
        'ROUGE-L': sum_res['rougeL'],
        'QA F1': qa_res['f1'],
        'QA EM': qa_res['exact_match'],
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
# ── NOTE: For per-task adapters, we load each adapter and eval its specific task
# Here we combine scores for a unified comparison row
# You can also eval each task adapter separately for cleaner per-task analysis

# Qwen3-8B fine-tuned (using translation adapter as example of combined eval)
# For full rigor: load each task adapter separately and record its task metric

ADAPTERS = [
    ('iwasbinod/qwen3-8b-nepali-translation-qlora',    'Qwen3-8B FT Translation'),
    ('iwasbinod/qwen3-8b-nepali-summarization-qlora',  'Qwen3-8B FT Summarization'),
    ('iwasbinod/qwen3-8b-nepali-qa-qlora',             'Qwen3-8B FT QA'),
    ('iwasbinod/llama32-3b-nepali-translation-qlora',  'Llama-3.2-3B FT Translation'),
    ('iwasbinod/llama32-3b-nepali-summarization-qlora','Llama-3.2-3B FT Summarization'),
    ('iwasbinod/llama32-3b-nepali-qa-qlora',           'Llama-3.2-3B FT QA'),
]

for adapter_repo, label in ADAPTERS:
    try:
        all_results[label] = eval_adapter(adapter_repo, label, label)
        print(f'Results: {all_results[label]}')
    except Exception as e:
        print(f'[SKIP] {label}: {e}')

In [ ]:
# ── FINAL COMPARISON TABLE ───────────────────
df = build_results_table(all_results)
print('\n' + '='*70)
print('FINAL RESULTS: BASE vs FINE-TUNED')
print('='*70)
print(df.to_string(index=False))

df.to_csv('results/metrics_summary.csv', index=False)
print('\nSaved to results/metrics_summary.csv')

In [ ]:
# ── PLOT ────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ['BLEU', 'ROUGE-1', 'ROUGE-L', 'QA F1']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

models = list(all_results.keys())
colors = ['#e74c3c' if 'Base' in m else '#2ecc71' for m in models]

for ax, metric in zip(axes, metrics):
    values = [all_results[m].get(metric, 0) for m in models]
    bars = ax.bar(range(len(models)), values, color=colors)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Base vs Fine-tuned: Nepali NLP Tasks', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/base_vs_finetuned_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/plots/base_vs_finetuned_comparison.png')